# Image support

Images go into a user message as **image blocks**, next to text blocks. Claude answers with text.

```python
{"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": "<base64>"}}
```

| Limit | Value |
|---|---|
| images per request | up to 100, across all messages |
| size | 5 MB per image |
| dimensions | 8000 px max; 2000 px max when sending several images |
| source | base64 or a URL |
| cost | about `width * height / 750` tokens |

Prompting works as with text: a step-by-step method beats a bare question ("How many marbles?" can
give a wrong count), and a reference image with a known answer works as a one-shot example.

The course example: an insurance company rates the **fire risk** of a home from a satellite image instead of
sending an inspector. The seven images in [`images/`](./images/) are the course's.

## Setup

In [1]:
import base64
from pathlib import Path

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def image_block(path):
    image_bytes = base64.standard_b64encode(Path(path).read_bytes()).decode("utf-8")
    return {
        "type": "image",
        "source": {"type": "base64", "media_type": "image/png", "data": image_bytes},
    }


images = sorted(Path("images").glob("prop*.png"))
print([image.name for image in images])

['prop1.png', 'prop2.png', 'prop3.png', 'prop4.png', 'prop5.png', 'prop6.png', 'prop7.png']


## What do you see?

The lesson's first request: an image block, then a text block, in one user message.

In [2]:
messages = []
add_user_message(messages, [
    image_block("images/prop7.png"),
    {"type": "text", "text": "What do you see in this image?"},
])
print(text_from_message(chat(messages)))

# Aerial View Analysis

This is an aerial/overhead photograph showing:

**Main Features:**
- **A residential building** with a gray/white roof in the center, surrounded by dense forest
- **Dense tree coverage** in green and dark green tones covering most of the image
- **What appears to be a clearing** around the structure
- The building looks relatively isolated, nestled deep within a wooded area

**Context:**
This appears to be taken from a drone, helicopter, or high-altitude aircraft. The image quality suggests it may be from satellite imagery or aerial mapping. The structure could be:
- A remote cabin or house
- A ranger station or research facility
- A hunting lodge
- A residential property in a rural/forested area

The heavy tree coverage and isolation suggest this is likely in a remote, forested region with significant woodland development.


## What an image costs

The formula against the real count: the same request with and without the image, the difference in
`input_tokens`. The size comes from the PNG header (bytes 16-24).

In [3]:
import struct


def png_size(path):
    return struct.unpack(">II", Path(path).read_bytes()[16:24])


def input_tokens(content):
    # client.messages.count_tokens would be free, but OpenRouter answers 404: a 1-token request instead
    response = client.messages.create(model=model, max_tokens=1, messages=[{"role": "user", "content": content}])
    return response.usage.input_tokens


text_only = input_tokens([{"type": "text", "text": "Describe."}])
for path in images:
    width, height = png_size(path)
    measured = input_tokens([image_block(path), {"type": "text", "text": "Describe."}]) - text_only
    print(f"{path.name}  {width:4}x{height:<4}  formula {width * height // 750:5}  measured {measured:5}")

prop1.png   928x1226  formula  1516  measured  1500


prop2.png  1062x1134  formula  1605  measured  1562


prop3.png  1062x1154  formula  1634  measured  1562


prop4.png  1256x1790  formula  2997  measured  1555


prop5.png  1148x954   formula  1460  measured  1439


prop6.png  1392x808   formula  1499  measured  1454


prop7.png   934x1256  formula  1564  measured  1534


## Fire risk: a bare question vs the course prompt

All seven properties, each prompt **three times**: a good prompt should give the same rating every time. The course prompt gets one extra line, `RATING: N`, so the number is easy to read off. 42 requests, about a minute.

In [4]:
import re

SIMPLE_PROMPT = "What is the fire risk of this home on a scale from 1 (low) to 4 (severe)? End with a line \"RATING: N\"."

FIRE_PROMPT = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75%+)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the home
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple vulnerability points
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numerical rating.
End with a line in the form "RATING: N".
"""


def rate(path, prompt):
    messages = []
    add_user_message(messages, [image_block(path), {"type": "text", "text": prompt}])
    response = chat(messages, max_tokens=2000)
    text = text_from_message(response)
    match = re.search(r"RATING:\s*([1-4])", text)
    return (match.group(1) if match else "?"), text


RUNS = 3
answers = {}
stable = {"simple": 0, "course": 0}
for path in images:
    simple = [rate(path, SIMPLE_PROMPT)[0] for _ in range(RUNS)]
    course = [rate(path, FIRE_PROMPT) for _ in range(RUNS)]
    answers[path.name] = course[0][1]
    course = [rating for rating, _ in course]
    stable["simple"] += len(set(simple)) == 1
    stable["course"] += len(set(course)) == 1
    print(f"{path.name}  simple: {' '.join(simple)}   course prompt: {' '.join(course)}")

print(f"\nsame rating in all {RUNS} runs: simple {stable['simple']}/{len(images)}, course prompt {stable['course']}/{len(images)}")

prop1.png  simple: 3 3 3   course prompt: 4 4 4


prop2.png  simple: 3 3 3   course prompt: 3 3 3


prop3.png  simple: 2 3 3   course prompt: 3 3 3


prop4.png  simple: 3 2 2   course prompt: 2 3 2


prop5.png  simple: 3 3 3   course prompt: 3 4 3


prop6.png  simple: 3 3 3   course prompt: 4 3 4


prop7.png  simple: 3 3 3   course prompt: 4 4 4

same rating in all 3 runs: simple 5/7, course prompt 4/7


The bare question looks more stable, but only because it answers 3 almost every time (18 of 21):
the middle of the scale. The course prompt uses the whole scale: prop1 (branches over the roof) and
prop7 (a house inside a continuous forest) get 4 in every run. It wavers between 3 and 4, or 2 and 3,
on the borderline properties.

Stable is not the same as right. There is no ground truth here, so this is not an accuracy score;
to get one, label the images by hand and run an eval, as in module 2. A one-shot example (a labelled
reference image before the target one) is the next thing to try on the borderline cases.

The full answer for one property, to see the five steps:

In [5]:
print(answers["prop1.png"])

# Satellite Image Fire Risk Analysis

1. **Residence Identification:** The primary residence is the tan/beige colored structure with a peaked roof located in the center of the property, clearly distinguishable as the main dwelling among the surrounding forest.

2. **Tree Overhang Analysis:** Extensive coniferous tree canopy surrounds and directly overhangs the residence, with dense dark green foliage covering an estimated 50-75% of the roof area, particularly concentrated on the north, east, and west sides of the structure.

3. **Fire Risk Assessment:** The overhanging trees create significant vulnerability to wildfire, with continuous fuel pathways from the surrounding dense forest directly to the roof, and multiple points where branch-laden vegetation forms bridges between wildland vegetation and the structure.

4. **Defensible Space Identification:** The property exhibits poor defensible space with no visible separation between the structure and surrounding forest—tree canopies are 

Next lesson: PDF support.